In [6]:
import argparse
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from crop_embed.models.fp_head_model import (
    MLPModel, LinearModel, FPSumHeadModel,
)
from crop_embed import FixedWindowEmbedder, MetricLogger, metrics_path_for
from crop_embed.data.loading import prepare_data
from crop_embed.train import masked_mse, _compute_metrics


ModuleNotFoundError: No module named 'crop_embed.models'

In [ ]:

# ── CLI ───────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Local JSONL metrics sidecar (always on) + optional wandb. See
# crop_embed/logging_utils.py and notebooks/track_training.ipynb.
logger = MetricLogger(
    metrics_path_for(args.output),
    wandb_project=args.wandb_project,
    wandb_name=args.wandb_name,
    config=vars(args),
)
print(f"Logging metrics to {logger.metrics_path}")

# ── Dataset, targets, and the shared train/val split ──────────────────────────
# Generate the split first with: python scripts/cache_split.py --output <SPLIT_PATH>

SPLIT_PATH = "splits/sativas413_seed42.pt"
data = prepare_data(split_path=SPLIT_PATH)
dataset    = data["dataset"]
Y          = data["Y"]
trait_cols = data["trait_cols"]
train_idx  = data["train_idx"]
val_idx    = data["val_idx"]

# ── 1. Load fixed window cache ────────────────────────────────────────────────

print(f"\nLoading cache from {args.cache} …")
embedder = FixedWindowEmbedder.from_file(args.cache, dataset)
cache           = embedder.cache.float()       # (n_fps, D)
sample_fp_index = embedder.sample_fp_index      # (n_samples, n_windows)
emb_dim  = cache.shape[1]
n_traits = Y.shape[1]

# The cache's sample→fingerprint index must line up with the dataset the split
# was built from; otherwise train_idx/val_idx point at the wrong samples.
if (sample_fp_index.shape != dataset.sample_fp_index.shape
        or not torch.equal(sample_fp_index, dataset.sample_fp_index)):
    raise SystemExit(
        "Cache sample→fingerprint index doesn't match the dataset built from the "
        "split's VCF/windowing. Regenerate the cache for this windowing."
    )
print(f"  {cache.shape[0]:,} fingerprints × {emb_dim} dims; {sample_fp_index.shape[0]} samples")

# ── 1.a Pre-sum into one embedding per sample ─────────────────────────────────
# Frozen cache → the per-sample sum is constant, so compute it once. embedding_bag
# fuses the gather+sum so the (n_samples, n_windows, D) intermediate never exists.

summed = F.embedding_bag(sample_fp_index, cache, mode="sum")   # (n_samples, D)
meaned = F.embedding_bag(sample_fp_index, cache, mode="mean")   # (n_samples, D)

train_ds = TensorDataset(summed[train_idx], Y[train_idx])
val_x = summed[val_idx].to(device)
val_y = Y[val_idx].to(device)
train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)